# Recover the first learning-gate failure without restarting training

The first full heldout measurement failed because notebook 03 indexed gate history with `[-500]` instead of retaining `[-500:]`. Notebook 05 now also resets a resumed controller's lifecycle status. This notebook permits **only those two declared source changes** and preserves the committed 242-step model, Adam state, scheduler, scaler, every RNG, sampler, replay, historical opponents and any in-flight game. Original start and deadline timestamps remain unchanged.

**Inputs:** immutable prior snapshot under `data/runtime_recovery/gate-history-slice/prior`, its `archive.json`, and current source-bound recovery/first-gate regression receipts. **Outputs:** read-only readiness, staged delivery/protocol/checkpoints, a durable promotion journal and migration evidence; optionally one detached tmux restart. **Runtime:** read-only checks normally under 30 seconds; opt-in validation and recovery approximately 1–3 minutes. **Checkpoint location:** the original `checkpoints/ahr-native-v5-main/` and controller directory, with displaced originals preserved in the recovery archive. **Single writer:** the exact main tmux session must be dead; stable migration/protocol locks and both old/new checkpoint locks protect promotion.

**Normal Run All is read-only for the run.** To commit the authorized recovery, run this notebook in a fresh `ahr-go` kernel with `AHR_NATIVE_V5_RECOVER=1`. If interrupted, repeat the same command: the journal resumes the prepared promotion rather than generating a new experiment. After COMMITTED, the notebook verifies the current lineage and starts it only if its exact tmux session is absent or dead. Detach with Ctrl-b, then d. No learning budget or seed is reset.

In [ ]:
from pathlib import Path
import os, nbformat
PROJECT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'notebooks/00_instance_setup_and_recovery.ipynb').is_file())
book=nbformat.read(PROJECT/'experiments/ahr_native_v5/00_shared.ipynb',4)
for cell in book.cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'v5 shared','exec'),globals())
_ = v5_load_v4(PROJECT,globals())
for name in V5_NOTEBOOKS[1:]:_ = v5_import_exports(PROJECT/V5_ROOT/name,globals())

## Verify the immutable snapshot and permitted source changes

The prior validation hash, protocol digest, all 24 exported source hashes and every retained checkpoint payload are checked. The alternate old notebook 03 is used only if its exact exported hash matches the archived protocol; the prior snapshot is never modified.

In [ ]:
import contextlib, hashlib, io, json, shutil, subprocess, time
from zoneinfo import ZoneInfo

V5_GATE_RECOVERY_ROOT='experiments/ahr_native_v5/data/runtime_recovery/gate-history-slice'
V5_GATE_RECOVERY_ID='gate-history-slice'
V5_GATE_ALLOWED_NEW={
 'experiments/ahr_native_v5/03_learning.ipynb':'6c5179db1adf8e886ac2115004400618e170c5250afce752728fce9b70560ee0',
 'experiments/ahr_native_v5/05_budgeted_pipeline.ipynb':'f752b2d5764b9415c00c9aded5429b5f3b5d4afdd2a7dc485bfa87ecb3450904'}
V5_GATE_PROOFS=('training_recovery','learning_recovery_extensions','curriculum_recovery','arena_validation','controller_recovery','first_gate_regression')

def v5_gate_checkpoint_inventory(directory):
    directory=Path(directory);manifest=v5_read_json(directory/'manifest.json');entries=manifest.get('entries',[])
    if len(entries)<2 or len({e['file'] for e in entries})!=len(entries):raise RuntimeError('Retain latest two distinct valid checkpoints')
    if manifest.get('latest')!=max(entries,key=lambda e:e['global_step'])['file'] or manifest.get('best') not in {e['file'] for e in entries}:raise RuntimeError('Invalid latest/best pointers')
    payloads={}
    for entry in entries:
        state=_checkpoint_entry_state(directory,entry)
        if state is None:raise RuntimeError('Corrupt retained checkpoint: '+str(directory/entry['file']))
        payloads[entry['file']]=state
    return manifest,payloads

def v5_gate_archive_evidence(project):
    project=Path(project);root=project/V5_GATE_RECOVERY_ROOT;prior=root/'prior'
    archived=v5_read_json(root/'archive.json');protocol=v5_read_json(prior/V5_PROTOCOL)
    if file_sha256(prior/V5_PROTOCOL)!=archived['prior_protocol_sha256']:raise RuntimeError('Prior protocol file changed')
    core={k:v for k,v in protocol.items() if k!='configuration_sha256'}
    if protocol['configuration_sha256']!=v5_digest(core) or protocol['configuration_sha256']!=archived['prior_configuration_sha256']:raise RuntimeError('Prior protocol digest differs')
    old_validation=prior/'reports/ahr_native_v5/delivery_validation.json'
    if file_sha256(old_validation)!=protocol['validation_sha256']:raise RuntimeError('Prior immutable delivery validation changed')
    validation=v5_read_json(old_validation)
    if validation['status']!='PASS' or validation['sources']!=protocol['sources'] or archived['prior_sources']!=protocol['sources'] or len(protocol['sources'])!=24:raise RuntimeError('Prior 24-source validation identity differs')
    overrides={}
    for relative,expected in protocol['sources'].items():
        source=prior/relative
        if v5_code_sha(source)!=expected:
            alternate=project/'experiments/ahr_native_v5/data/runtime_migrations/first-gate-1790960103/old_03_learning.ipynb'
            if relative!='experiments/ahr_native_v5/03_learning.ipynb' or not alternate.is_file() or v5_code_sha(alternate)!=expected:raise RuntimeError('Archived source does not match prior protocol: '+relative)
            overrides[relative]=str(alternate.relative_to(project))
    current=v5_source_identity(project)
    changes={p:{'before':protocol['sources'][p],'after':current[p]} for p in current if current[p]!=protocol['sources'].get(p)}
    if set(current)!=set(protocol['sources']) or set(changes)!=set(V5_GATE_ALLOWED_NEW) or any(current[p]!=sha for p,sha in V5_GATE_ALLOWED_NEW.items()):raise RuntimeError('Only the declared 03 history-slice and 05 lifecycle-status source changes are authorized')
    main_manifest,main=v5_gate_checkpoint_inventory(prior/'checkpoints'/V5_TRAIN_RUN)
    controller_manifest,controller=v5_gate_checkpoint_inventory(prior/'checkpoints'/V5_CONTROLLER)
    expected_training=asdict(V5TrainConfig(**protocol['training']))
    for state in main.values():
        if state['identity']['sources']!=protocol['sources'] or state['config']!=expected_training or state['purpose']!='go_training':raise RuntimeError('Prior training identity/configuration differs')
    latest=main[main_manifest['latest']]
    if latest['optimizer_step']!=242 or latest['global_step']!=242:raise RuntimeError('Expected the original committed 242-step native lineage')
    for state in controller.values():
        if state['config']!=protocol:raise RuntimeError('Prior controller protocol differs')
    clocks={k:controller[controller_manifest['latest']]['metrics'][k] for k in ('started_unix','learning_deadline_unix','target_deadline_unix','hard_deadline_unix')}
    return {'root':root,'prior':prior,'protocol':protocol,'sources':current,'changes':changes,'overrides':overrides,
        'main_manifest':main_manifest,'main':main,'controller_manifest':controller_manifest,'controller':controller,'clocks':clocks}

def v5_gate_proof_evidence(project,sources):
    evidence={};problems=[]
    for name in V5_GATE_PROOFS:
        path=v5_report_path(project,name+'.json')
        if not path.is_file():problems.append(name+': missing');continue
        proof=v5_read_json(path);identity=proof.get('sources',proof.get('source_identity'))
        if proof.get('status')!='PASS' or identity!=sources:problems.append(name+': stale or failed');continue
        if name in ('training_recovery','curriculum_recovery','arena_validation') and proof.get('exact_resume') is not True:problems.append(name+': exact recovery not proved');continue
        if not proof.get('checks') or not all(proof['checks'].values()):problems.append(name+': incomplete checks');continue
        evidence[name]={'path':str(path.relative_to(project)),'sha256':file_sha256(path),'proof':proof}
    return evidence,problems

def v5_gate_main_session(project,require_dead=False):
    exists=subprocess.run(['tmux','has-session','-t','='+V5_SESSION],capture_output=True).returncode==0
    panes=[]
    if exists:
        result=subprocess.run(['tmux','list-panes','-s','-t','='+V5_SESSION,'-F','#{pane_dead} #{pane_pid}'],capture_output=True,text=True,check=True)
        panes=[{'dead':line.split()[0]=='1','pid':int(line.split()[1])} for line in result.stdout.splitlines() if line.strip()]
        if not panes:raise RuntimeError('Cannot inspect the exact AHR main session')
    active=exists and any(not pane['dead'] for pane in panes)
    if require_dead and active:raise RuntimeError('The exact AHR main tmux session is still active; no migration may mutate it')
    if require_dead:
        listing=subprocess.check_output(['ps','-eo','pid,args'],text=True)
        for line in listing.splitlines():
            if 'jupyter-nbconvert' in line and '05_budgeted_pipeline.ipynb' in line and not line.lstrip().startswith(str(os.getpid())+' '):raise RuntimeError('An AHR controller process remains active outside the dead session')
    return {'exists':exists,'active':active,'panes':panes}

def v5_gate_recovery_readiness(project):
    evidence=v5_gate_archive_evidence(project);proofs,problems=v5_gate_proof_evidence(project,evidence['sources'])
    journal=evidence['root']/'journal.json'
    state=v5_read_json(journal)['state'] if journal.exists() else 'NOT_PREPARED'
    session=v5_gate_main_session(project)
    return {'status':'READY' if not problems else 'WAITING_FOR_CURRENT_PROOFS','mode':'READ_ONLY','journal_state':state,
        'sources':evidence['sources'],'permitted_changes':evidence['changes'],'archived_source_overrides':evidence['overrides'],
        'optimizer_step_preserved':242,'clocks':evidence['clocks'],'proofs':{k:{'path':v['path'],'sha256':v['sha256']} for k,v in proofs.items()},
        'problems':problems,'exact_session':session,'scientific_state_changed':False}


## Prepare verified replacements and a durable journal

The new delivery validation runs the current notebook 07 validation prefix before its existing-protocol branch. The gate regression is required explicitly. Training payloads change only `identity.sources`; controller payloads change only their protocol and documented lifecycle status/error. Persisted payloads are loaded back and compared with the prior snapshots before any canonical path is replaced.

In [ ]:
def v5_gate_fresh_delivery(project,evidence):
    proofs,problems=v5_gate_proof_evidence(project,evidence['sources'])
    if problems:raise RuntimeError('Recovery proofs are not current: '+', '.join(problems))
    book=nbformat.read(Path(project)/V5_ROOT/'07_validate_and_launch.ipynb',4)
    candidates=[c.source for c in book.cells if c.cell_type=='code' and 'curriculum=v5_curriculum' in c.source]
    if len(candidates)!=1:raise RuntimeError('Ambiguous notebook07 validation cell')
    source=candidates[0];marker="path=v5_report_path(PROJECT,'delivery_validation.json')"
    if marker not in source:raise RuntimeError('Notebook07 validation boundary changed')
    namespace=dict(globals());namespace['PROJECT']=Path(project)
    exec(compile(source.split(marker,1)[0],'07 current validation prefix for explicit recovery','exec'),namespace)
    receipt=namespace['receipt'];gate=proofs['first_gate_regression']['proof']
    if receipt['status']!='PASS' or receipt['sources']!=evidence['sources']:raise RuntimeError('Fresh delivery identity differs')
    receipt['checks']['learning_gate_runtime']='PASS';receipt['first_gate_regression']=gate
    receipt['runtime_recovery']={'id':V5_GATE_RECOVERY_ID,'prior_configuration_sha256':evidence['protocol']['configuration_sha256'],
        'reason':'Explicit history-slice and controller-lifecycle correction; scientific lineage and original clocks retained'}
    path=v5_report_path(project,'recovery_delivery_validation.json');v5_atomic_json(path,receipt)
    return receipt,path

def v5_gate_fsync_directory(path):
    fd=os.open(str(Path(path)),os.O_DIRECTORY)
    try:os.fsync(fd)
    finally:os.close(fd)

def v5_gate_copy_atomic(source,target):
    atomic_write_bytes(Path(target),Path(source).read_bytes())

def v5_gate_stage_checkpoints(evidence,new_protocol,stage):
    records={};checks={}
    for run in (V5_TRAIN_RUN,V5_CONTROLLER):
        kind='main' if run==V5_TRAIN_RUN else 'controller';manifest=copy.deepcopy(evidence[kind+'_manifest'])
        destination=Path(stage)/'checkpoints'/run;destination.mkdir(parents=True,exist_ok=True)
        updates=[]
        with checkpoint_lock(destination):
            for entry in manifest['entries']:
                original=evidence[kind][entry['file']];changed=copy.deepcopy(original)
                if kind=='main':
                    changed['identity']['sources']=copy.deepcopy(evidence['sources'])
                    comparable=copy.deepcopy(changed);comparable['identity']['sources']=copy.deepcopy(original['identity']['sources'])
                else:
                    changed['config']=copy.deepcopy(new_protocol)
                    changed['metrics']['status']='RUNNING';changed['metrics']['last_error']=None
                    comparable=copy.deepcopy(changed);comparable['config']=copy.deepcopy(original['config'])
                    comparable['metrics']['status']=original['metrics']['status'];comparable['metrics']['last_error']=original['metrics']['last_error']
                if not nested_equal(original,comparable):raise RuntimeError('Undeclared scientific checkpoint change: '+run+'/'+entry['file'])
                buffer=io.BytesIO();torch.save(changed,buffer);atomic_write_bytes(destination/entry['file'],buffer.getvalue())
                updates.append({**entry,'sha256':file_sha256(destination/entry['file'])})
                loaded=torch.load(destination/entry['file'],map_location='cpu',weights_only=False)
                if not nested_equal(loaded,changed):raise RuntimeError('Persisted migrated checkpoint differs')
                checks[run+'/'+entry['file']]=True
            manifest['entries']=updates;v5_atomic_json(destination/'manifest.json',manifest)
        validated,loaded=v5_gate_checkpoint_inventory(destination)
        if validated['latest']!=evidence[kind+'_manifest']['latest'] or validated['best']!=evidence[kind+'_manifest']['best']:raise RuntimeError('Migration changed checkpoint cursors')
        records[run]={'manifest_sha256':file_sha256(destination/'manifest.json'),'entries':{e['file']:e['sha256'] for e in validated['entries']},
            'old_manifest_sha256':file_sha256(evidence['prior']/'checkpoints'/run/'manifest.json')}
    return records,checks

def v5_gate_prepare_recovery(project,evidence):
    root=evidence['root'];journal_path=root/'journal.json'
    if journal_path.exists():return v5_read_json(journal_path)
    receipt,delivery_path=v5_gate_fresh_delivery(project,evidence)
    stage=root/'staged';stage.mkdir(parents=True,exist_ok=True)
    staged_delivery=stage/'reports/ahr_native_v5/delivery_validation.json';v5_gate_copy_atomic(delivery_path,staged_delivery)
    new_protocol=copy.deepcopy(evidence['protocol'])
    new_protocol['sources']=copy.deepcopy(evidence['sources']);new_protocol['validation_sha256']=file_sha256(staged_delivery)
    new_protocol['runtime_migration']={'id':V5_GATE_RECOVERY_ID,'prior_configuration_sha256':evidence['protocol']['configuration_sha256'],
        'prior_protocol_sha256':file_sha256(evidence['prior']/V5_PROTOCOL),'prior_validation_sha256':evidence['protocol']['validation_sha256'],
        'source_changes':copy.deepcopy(evidence['changes']),'scientific_payload_changes':'Training: identity.sources only. Controller: protocol plus RUNNING/clear-error lifecycle only.',
        'original_clocks':copy.deepcopy(evidence['clocks']),'migrated_at_utc':v5_now()}
    new_protocol.pop('configuration_sha256');new_protocol['configuration_sha256']=v5_digest(new_protocol)
    staged_protocol=stage/V5_PROTOCOL;v5_atomic_json(staged_protocol,new_protocol)
    records,checks=v5_gate_stage_checkpoints(evidence,new_protocol,stage)
    allowed={'sources','validation_sha256','configuration_sha256','runtime_migration'}
    if any(not nested_equal(evidence['protocol'][key],new_protocol[key]) for key in evidence['protocol'] if key not in allowed):raise RuntimeError('Migration changed training or budget protocol')
    resources=[{'name':'training_checkpoints','relative':'checkpoints/'+V5_TRAIN_RUN,'kind':'directory','new_sha256':records[V5_TRAIN_RUN]['manifest_sha256'],'old_sha256':records[V5_TRAIN_RUN]['old_manifest_sha256'],'entries':records[V5_TRAIN_RUN]['entries']},
        {'name':'controller_checkpoints','relative':'checkpoints/'+V5_CONTROLLER,'kind':'directory','new_sha256':records[V5_CONTROLLER]['manifest_sha256'],'old_sha256':records[V5_CONTROLLER]['old_manifest_sha256'],'entries':records[V5_CONTROLLER]['entries']},
        {'name':'delivery_validation','relative':'reports/ahr_native_v5/delivery_validation.json','kind':'file','new_sha256':file_sha256(staged_delivery),'old_sha256':evidence['protocol']['validation_sha256']},
        {'name':'protocol','relative':V5_PROTOCOL,'kind':'file','new_sha256':file_sha256(staged_protocol),'old_sha256':file_sha256(evidence['prior']/V5_PROTOCOL)}]
    journal={'schema':'ahr_native_v5_explicit_runtime_migration','id':V5_GATE_RECOVERY_ID,'state':'PREPARED','prepared_at_utc':v5_now(),
        'prior_configuration_sha256':evidence['protocol']['configuration_sha256'],'new_configuration_sha256':new_protocol['configuration_sha256'],
        'sources':evidence['sources'],'resources':resources,'completed_resources':[],
        'checks':{'all_retained_payloads_scientifically_equal':all(checks.values()),'latest_two_and_best_preserved':True,'original_training_configuration_preserved':True,'original_budget_and_clocks_preserved':True},
        'payload_checks':checks,'original_clocks':evidence['clocks'],'optimizer_step':242,
        'lifecycle_changes':'Each retained controller checkpoint is RUNNING with last_error=None; global steps, all other metrics and every RNG unchanged.'}
    v5_atomic_json(journal_path,journal);return journal


## Commit once, recover interrupted promotion, then resume the same run

PREPARED → COMMITTING → COMMITTED is written durably. Each resource is recognized by its exact old or new hash; an unexpected resource stops recovery. Displaced canonical files remain in the archive. The strict current-source loaders and original clocks are verified before restarting the exact AHR tmux session. Re-running after commitment inspects the resumed lineage rather than restoring an older model.

In [ ]:
def v5_gate_resource_matches(path,resource,new):
    path=Path(path);expected=resource['new_sha256' if new else 'old_sha256']
    if resource['kind']=='file':return path.is_file() and file_sha256(path)==expected
    if not (path/'manifest.json').is_file() or file_sha256(path/'manifest.json')!=expected:return False
    try:
        manifest,payloads=v5_gate_checkpoint_inventory(path)
        if new and {e['file']:e['sha256'] for e in manifest['entries']}!=resource['entries']:return False
        return True
    except (RuntimeError,ValueError,KeyError):return False

def v5_gate_promote_resource(project,root,resource):
    current=Path(project)/resource['relative'];prepared=root/'staged'/resource['relative'];displaced=root/'displaced'/resource['relative']
    if v5_gate_resource_matches(current,resource,True):return
    if current.exists():
        if not v5_gate_resource_matches(current,resource,False):raise RuntimeError('Canonical resource is neither verified old nor prepared new: '+resource['relative'])
        if displaced.exists():raise RuntimeError('Old canonical resource reappeared after displacement')
        displaced.parent.mkdir(parents=True,exist_ok=True);os.replace(current,displaced)
        v5_gate_fsync_directory(current.parent);v5_gate_fsync_directory(displaced.parent)
    if not v5_gate_resource_matches(displaced,resource,False):raise RuntimeError('Missing valid displaced original: '+resource['relative'])
    if not v5_gate_resource_matches(prepared,resource,True):raise RuntimeError('Missing verified prepared replacement: '+resource['relative'])
    current.parent.mkdir(parents=True,exist_ok=True);os.replace(prepared,current)
    v5_gate_fsync_directory(current.parent);v5_gate_fsync_directory(prepared.parent)
    if not v5_gate_resource_matches(current,resource,True):raise RuntimeError('Promoted resource failed verification')

def v5_gate_verify_committed(project,evidence,journal,allow_progress=False):
    protocol=v5_protocol_read(project)
    if protocol['configuration_sha256']!=journal['new_configuration_sha256'] or protocol['sources']!=evidence['sources']:raise RuntimeError('Committed protocol identity differs')
    current=V5TrainConfig(**protocol['training']);curriculum=v5_curriculum(project)
    prior_entries,prior=v5_training_prior(project,current);identity=v5_training_identity(project,current,curriculum,prior)
    main,entry=v5_read_training_checkpoint(Path(project)/'checkpoints'/V5_TRAIN_RUN,identity)
    if main is None or main['optimizer_step']<242 or (not allow_progress and main['optimizer_step']!=242):raise RuntimeError('Committed training step differs')
    controller=v5_controller_load(project,protocol)
    if controller is None:raise RuntimeError('Committed controller missing')
    clocks={key:controller['metrics'][key] for key in journal['original_clocks']}
    if clocks!=journal['original_clocks']:raise RuntimeError('Original controller clock was reset')
    if not allow_progress:
        old=evidence['main'][evidence['main_manifest']['latest']];comparable=copy.deepcopy(main);comparable['identity']['sources']=old['identity']['sources']
        if not nested_equal(old,comparable):raise RuntimeError('Canonical training scientific state changed during promotion')
    return {'status':'PASS','optimizer_step':main['optimizer_step'],'global_step':main['global_step'],
        'checkpoint_sha256':entry['sha256'],'clocks':clocks,'configuration_sha256':protocol['configuration_sha256'],
        'strict_current_source_training_loader':'PASS','strict_controller_loader':'PASS','original_budget_preserved':True}

def v5_recover_gate_fix(project,launch=True):
    project=Path(project);evidence=v5_gate_archive_evidence(project);root=evidence['root'];journal_path=root/'journal.json'
    stable_lock=project/'checkpoints'/'ahr-native-v5-runtime-migration'
    with checkpoint_lock(stable_lock),checkpoint_lock(project/'checkpoints'/'ahr-native-v5-protocol'):
        journal=v5_read_json(journal_path) if journal_path.exists() else None
        if journal is not None and journal.get('sources')!=evidence['sources']:raise RuntimeError('Prepared recovery source identity changed')
        if journal is None or journal['state']!='COMMITTED':
            v5_gate_main_session(project,require_dead=True)
            if journal is None:journal=v5_gate_prepare_recovery(project,evidence)
            if journal['state'] not in ('PREPARED','COMMITTING'):raise RuntimeError('Unknown recovery journal state')
            # Lock every existing old, staged and displaced checkpoint directory. Locks move with renamed directories.
            with contextlib.ExitStack() as locks:
                directories=[]
                for run in (V5_TRAIN_RUN,V5_CONTROLLER):
                    for parent in (project,root/'staged',root/'displaced'):
                        path=parent/'checkpoints'/run
                        if path.is_dir():directories.append(path)
                for path in sorted(set(directories)):locks.enter_context(checkpoint_lock(path))
                journal['state']='COMMITTING';v5_atomic_json(journal_path,journal)
                for resource in journal['resources']:
                    v5_gate_promote_resource(project,root,resource)
                    if resource['name'] not in journal['completed_resources']:journal['completed_resources'].append(resource['name'])
                    v5_atomic_json(journal_path,journal)
                verified=v5_gate_verify_committed(project,evidence,journal)
                journal['state']='COMMITTED';journal['committed_at_utc']=v5_now();journal['committed_verification']=verified
                v5_atomic_json(journal_path,journal)
        verified=v5_gate_verify_committed(project,evidence,journal,allow_progress=True)
        session=v5_gate_main_session(project)
        launch_receipt=None
        if launch and not session['active']:
            v5_gate_main_session(project,require_dead=True)
            if session['exists']:subprocess.run(['tmux','kill-session','-t','='+V5_SESSION],check=True)
            launch_receipt=v5_launch_detached(project)
        result={'status':'COMMITTED_AND_RUNNING' if launch and (session['active'] or launch_receipt) else 'COMMITTED',
            'journal_state':journal['state'],'verification':verified,'scientific_state_preserved':True,
            'original_clocks':journal['original_clocks'],'lifecycle_changes':journal['lifecycle_changes'],
            'migration_journal':str(journal_path.relative_to(project)),'displaced_originals':str((root/'displaced').relative_to(project)),
            'launch':launch_receipt,'attach':'tmux attach -t '+V5_SESSION,'detach':'Ctrl-b, then d','updated_at_utc':v5_now()}
        v5_atomic_json(v5_report_path(project,'runtime_gate_recovery.json'),result);return result


In [ ]:
if os.environ.get('AHR_NATIVE_V5_RECOVER')=='1':
    result=v5_recover_gate_fix(PROJECT,launch=True)
else:
    result=v5_gate_recovery_readiness(PROJECT)
    v5_atomic_json(v5_report_path(PROJECT,'runtime_gate_recovery_readiness.json'),result)
from IPython.display import display, Markdown
print('Gate recovery:',result['status'],'| Journal:',result.get('journal_state'))
_clocks=result.get('original_clocks',result.get('clocks',{}))
_zone=ZoneInfo('Europe/Dublin')
_display_rows=[(key,dt.datetime.fromtimestamp(value,dt.timezone.utc).astimezone(_zone).strftime('%Y-%m-%d %H:%M %Z')) for key,value in _clocks.items()]
display(Markdown('| Preserved clock | Dublin time |\n|---|---|\n'+'\n'.join('| '+key.replace('_',' ')+' | '+value+' |' for key,value in _display_rows)))
if result.get('problems'):display(Markdown('Current proofs still needed: '+', '.join(result['problems'])))

The optimizer resumes from the latest committed step rather than recreating its first 242 updates. Work performed after the last checkpoint before the exception is replayed from the saved RNG and sampler. The first actual gate regression now checks full heldout prediction and a measured FAIL without treating that failure as a training success. Adaptive tactical learning continues until the declared competence and teacher checks pass or the original budget ends.